In [1]:
import os, sys, time, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
sys.path.insert(0, os.getcwd())
try:
    from 集成公共函数 import *
except ImportError as e:
    print('导入 集成公共函数.py 失败:', e)
    print('请确认Jupyter内核的工作目录为「实验室」文件夹(该文件与本notebook同目录)。')
    raise

from IPython.display import display


In [2]:
#区间预测参数
ALPHA        = 0.10     
UNC_METHOD   = 'std'     # 模型分歧度量: 'std'=成员预测标准差 / 'iqr'=四分位距(q75-q25)
NOISE_MODE   = 'run'   
N_SPLITS_CAL = 5         # 交叉共形覆盖率的折数

OUT_DIR      = PI_VQ_DIR

In [3]:
ART = load_or_build_models()         
ENSEMBLES     = ART['ENSEMBLES']
TUNED         = ART['TUNED']
W_raw         = ART['W_raw']
WEIGHTS_TABLE = ART['WEIGHTS_TABLE']
CV_SCORES     = ART['CV_SCORES']
BEST_PARAMS   = ART['BEST_PARAMS']
TUNE_META     = ART['TUNE_META']


print('已加载目标:', list(ENSEMBLES.keys()))
print('成员数:', len(base_learners()), '| 各目标非零权重成员数:',
      {(TAG_CN[k], int((W_raw[k] > 0.001).sum())) for k in ENSEMBLES})

[加载] 已从存档载入集成模型: d:\Code_File\Python_Data_AI\Project_CFPA_Teacher\Data\集成建模\集成模型存档.joblib (version=3)
已加载目标: ['TS', 'TM', 'FS', 'FM', 'IS']
成员数: 12 | 各目标非零权重成员数: {('拉伸强度', 4), ('冲击强度', 3), ('弯曲强度', 3), ('弯曲模量', 5), ('弹性模量', 4)}


In [4]:
#  构建嵌套OOF成员矩阵 + OOF残差诚
NESTED = ART.get('NESTED', None)
if NESTED is None:
    print('[提示] 存档中没有嵌套OOF结果')
    NESTED = {}
    for tag in SHEETS.values():
        t0 = time.time()
        NESTED[tag] = nested_oof_v2(tag, tuned=TUNED[tag], nested_tune=True,
                                    n_outer=N_OUTER, n_inner=N_INNER, verbose=False)
        print(f'  [{tag}] 完成 ({time.time()-t0:.0f}s)')
else:
    print('[复用] 已从模块一存档载入嵌套OOF结果')

[复用] 已从模块一存档载入嵌套OOF结果


In [5]:
EXT = load_external()

def cond_agg(values, cond):
    "试样级 -> 工况均值（按 cond 升序, 与 load_condition_means 对齐）"
    return pd.DataFrame(dict(v=np.asarray(values, float),
                             cond=np.asarray(cond))).groupby('cond')['v'].mean().values

def dispersion(Pm):
    "由成员预测矩阵算分歧度量"
    Pm = np.asarray(Pm, float)
    if UNC_METHOD == 'std':
        return np.std(Pm, axis=1)
    return np.quantile(Pm, 0.75, axis=1) - np.quantile(Pm, 0.25, axis=1)

def member_dispersion_cond(ensemble, d):
    Pm = ensemble.member_predictions(d[FEATS].values.astype(float))
    return cond_agg(dispersion(Pm), d['cond'].values)

CAL, U_EXT = {}, {}
for tag in SHEETS.values():
    nest = NESTED[tag]
    d = nest['d']
    cm = load_condition_means(tag)                  
    yc = cm['y'].values.astype(float)
    pc = cond_agg(nest['oof'], d['cond'].values)         #嵌套OOF -> 工况均值
    u_cal = member_dispersion_cond(ENSEMBLES[tag], d)
    ns = noise_scale(tag, mode=NOISE_MODE)                # σ_noise: 重复运行实验, 外部给定
    cmf = cm.copy()
    cmf['grp'] = group_labels(cmf)                      
    CAL[tag] = dict(y=yc, oof=pc, r=yc - pc, u=u_cal,
                    sigma_noise=ns['sigma'], sigma_source=ns['source'],
                    groups=cmf['grp'].values, cond=cm['cond'].values)

    Xe = EXT[tag][FEATS].values.astype(float)
    U_EXT[tag] = dispersion(ENSEMBLES[tag].member_predictions(Xe))
    print(f'  [{tag}] 校准集 n={len(yc)} 工况 | 残差SD={CAL[tag]["r"].std(ddof=1):.4g} | '
          f'σ_noise={ns["sigma"]:.4g}({ns["source"]}) | '
          f'σ_model中位数 校准={np.median(u_cal):.4g} / 外部={np.median(U_EXT[tag]):.4g}')

  [TS] 校准集 n=62 工况 | 残差SD=6.983 | σ_noise=3.068(sd_run) | σ_model中位数 校准=2.487 / 外部=3.019
  [TM] 校准集 n=62 工况 | 残差SD=1112 | σ_noise=519.5(sd_run) | σ_model中位数 校准=293.6 / 外部=341.9
  [FS] 校准集 n=62 工况 | 残差SD=9.151 | σ_noise=5.583(sd_run) | σ_model中位数 校准=4.038 / 外部=3.253
  [FM] 校准集 n=62 工况 | 残差SD=488.5 | σ_noise=268.8(sd_run) | σ_model中位数 校准=166.1 / 外部=178.5
  [IS] 校准集 n=62 工况 | 残差SD=2.504 | σ_noise=1.807(sd_run) | σ_model中位数 校准=0.6923 / 外部=0.7604


#### PICP / MPIW / PINAW

- **PICP**（覆盖率）$=\frac{1}{n}\sum_i \mathbb{1}[L_i\le y_i\le U_i]$ —— 越接近名义水平90%越好；
- **MPIW**（平均宽度）$=\frac{1}{n}\sum_i (U_i-L_i)$ —— PICP达标前提下越小越好；
- **PINAW**（归一化宽度）$=\text{MPIW}/R$ —— 跨目标可比。

In [6]:
# ----三种区间构造
METHODS = ['member_quantile', 'residual_conformal', 'variance_quadrature']
METHOD_CN = {'member_quantile': '成员分位数(分歧)',
             'residual_conformal': 'OOF残差共形(噪声, 常数宽度)',
             'variance_quadrature': '方差合成(分歧 噪声)'}
METHOD_COLORS = {'variance_quadrature': '#4C72B0',
                 'member_quantile': '#C44E52',
                 'residual_conformal': '#55A868'}

IV, CALIB, COV_TR, VD = {}, {}, {}, {}
for tag in SHEETS.values():
    d = NESTED[tag]['d']
    cond = d['cond'].values
    c = CAL[tag]
    ye = EXT[tag]['y_mean'].values.astype(float)
    mu_tr = c['oof']                                                
    Xe = EXT[tag][FEATS].values.astype(float)
    mu_ex = ENSEMBLES[tag].predict(Xe)
    Pm_tr = ENSEMBLES[tag].member_predictions(d[FEATS].values.astype(float))
    Pm_ex = ENSEMBLES[tag].member_predictions(Xe)

    # 1 成员分位数
    lo_s, hi_s = member_quantile_interval(Pm_tr, ALPHA)
    lo_tr1, hi_tr1 = cond_agg(lo_s, cond), cond_agg(hi_s, cond)
    lo_ex1, hi_ex1 = member_quantile_interval(Pm_ex, ALPHA)

    # 2 残差共形
    q_res = conformal_quantile(np.abs(c['r']), ALPHA)
    lo_tr2, hi_tr2, _ = residual_conformal_interval(mu_tr, c['r'], ALPHA)
    lo_ex2, hi_ex2, _ = residual_conformal_interval(mu_ex, c['r'], ALPHA)

    # 3 复合方差合成
    cal = calibrate_quadrature_conformal(c['r'], c['u'], c['sigma_noise'], ALPHA)
    CALIB[tag] = cal
    VD[tag] = variance_decomposition(c['r'], c['u'], c['sigma_noise'])   # 仅诊断, 不进区间
    lo_tr3, hi_tr3, _ = quadrature_conformal_interval(mu_tr, c['u'], cal)
    lo_ex3, hi_ex3, _ = quadrature_conformal_interval(mu_ex, U_EXT[tag], cal)

    IV[tag] = {
        'member_quantile': dict(tr=(lo_tr1, hi_tr1, mu_tr),
                                ex=(lo_ex1, hi_ex1, mu_ex), calib=None),
        'residual_conformal': dict(tr=(lo_tr2, hi_tr2, mu_tr),
                                   ex=(lo_ex2, hi_ex2, mu_ex),
                                   calib=dict(qhat=q_res, alpha=ALPHA)),
        'variance_quadrature': dict(tr=(lo_tr3, hi_tr3, mu_tr),
                                    ex=(lo_ex3, hi_ex3, mu_ex), calib=cal),
    }
    COV_TR[tag] = {
        'variance_quadrature': cross_conformal_coverage_quadrature(
            c['r'], c['u'], c['sigma_noise'], ALPHA,
            groups=c['groups'], n_splits=N_SPLITS_CAL),
        'residual_conformal': cross_conformal_coverage_constant(
            c['r'], ALPHA, groups=c['groups'], n_splits=N_SPLITS_CAL),
        'member_quantile': None,
    }
    
    print(f'  [{tag}] σ_noise={c["sigma_noise"]:.4g}({c["sigma_source"]}) | '
          f'q̂_合成={cal["qhat"]:.4g} | q̂_残差={q_res:.4g} | 残差SD={c["r"].std(ddof=1):.4g} | '
          f'κ={VD[tag]["kappa"]:.3g}')

  [TS] σ_noise=3.068(sd_run) | q̂_合成=2.558 | q̂_残差=11.05 | 残差SD=6.983 | κ=4.79
  [TM] σ_noise=519.5(sd_run) | q̂_合成=2.424 | q̂_残差=1976 | 残差SD=1112 | κ=4.74
  [FS] σ_noise=5.583(sd_run) | q̂_合成=2.178 | q̂_残差=15.32 | 残差SD=9.151 | κ=1.52
  [FM] σ_noise=268.8(sd_run) | q̂_合成=2.231 | q̂_残差=799.7 | 残差SD=488.5 | κ=3.57
  [IS] σ_noise=1.807(sd_run) | q̂_合成=1.916 | q̂_残差=4.142 | 残差SD=2.504 | κ=4.08


In [7]:
# --指标汇总
REC = 'variance_quadrature'
rows = []
for tag in SHEETS.values():
    c, cal = CAL[tag], CALIB[tag]
    yc = c['y']
    ye = EXT[tag]['y_mean'].values.astype(float)


    # 训练口径: 交叉共形 PICP 以及对应 MPIW/PINAW
    cov = COV_TR[tag][REC]
    tr = dict(PICP=cov['PICP'], MPIW=cov['MPIW'], PINAW=cov['MPIW'] / np.ptp(yc))
    # 外部口径: 真实折外(8个外部工况)
    lo, hi, mu = IV[tag][REC]['ex']
    ex = interval_metrics(ye, lo, hi)

    rows.append(dict(目标=TAG_CN[tag],
                     训练_PICP=round(tr['PICP'], 3),
                     训练_PICP_折间std=round(float(np.std(cov['PICP_folds'], ddof=1)), 3),
                     训练_MPIW=round(tr['MPIW'], 2),
                     训练_PINAW=round(tr['PINAW'], 3),
                     外部_PICP=round(ex['PICP'], 3),
                     外部_MPIW=round(ex['MPIW'], 2),
                     外部_PINAW=round(ex['PINAW'], 3),)
    )
metric_table = pd.DataFrame(rows)
metric_table['训练_PICP-名义'] = (metric_table['训练_PICP'] - (1 - ALPHA)).round(3)


print(f'指标汇总 方法: {METHOD_CN[REC]}')
print('  · 训练_PICP            = 交叉共形(折外)覆盖率')
print('  · 外部_PICP            = 8 个外部工况上的真实覆盖率')
display(metric_table)

指标汇总 方法: 方差合成(分歧 噪声)
  · 训练_PICP            = 交叉共形(折外)覆盖率
  · 外部_PICP            = 8 个外部工况上的真实覆盖率


,目标,训练_PICP,训练_PICP_折间std,训练_MPIW,训练_PINAW,外部_PICP,外部_MPIW,外部_PINAW,训练_PICP-名义
0,拉伸强度,0.901,0.137,21.84,0.493,1.00,22.63,1.181,0.001
1,弹性模量,0.901,0.092,3195.22,0.390,1.00,3095.00,1.014,0.001
2,弯曲强度,0.867,0.173,30.42,0.369,1.00,30.44,0.724,-0.033
3,弯曲模量,0.900,0.109,1515.31,0.388,0.75,1563.78,0.509,0.000
4,冲击强度,0.886,0.138,7.60,0.472,1.00,7.47,0.920,-0.014


In [8]:
# 三方法对比表
cmp_rows = []
for tag in SHEETS.values():
    c = CAL[tag]
    yc = c['y']
    ye = EXT[tag]['y_mean'].values.astype(float)
    for m in METHODS:
        if m == 'member_quantile':
            tr = interval_metrics(yc, *IV[tag][m]['tr'][:2])
        else:
            cov = COV_TR[tag][m]
            tr = dict(PICP=cov['PICP'], MPIW=cov['MPIW'],
                      PINAW=cov['MPIW'] / np.ptp(yc))
        ex = interval_metrics(ye, *IV[tag][m]['ex'][:2])
        cmp_rows.append(dict(目标=TAG_CN[tag], 方法=METHOD_CN[m],
                             训练_PICP=round(tr['PICP'], 3),
                             训练_MPIW=round(tr['MPIW'], 2),
                             外部_PICP=round(ex['PICP'], 3),
                             外部_MPIW=round(ex['MPIW'], 2),
                             外部_PINAW=round(ex['PINAW'], 3),))
compare_table = pd.DataFrame(cmp_rows)
display(compare_table)


,目标,方法,训练_PICP,训练_MPIW,外部_PICP,外部_MPIW,外部_PINAW
0,拉伸强度,成员分位数(分歧),0.887,9.31,0.750,10.97,0.573
1,拉伸强度,"OOF残差共形(噪声, 常数宽度)",0.919,22.58,1.000,22.11,1.154
2,拉伸强度,方差合成(分歧 噪声),0.901,21.84,1.000,22.63,1.181
3,弹性模量,成员分位数(分歧),0.839,1309.07,0.500,1332.29,0.437
4,弹性模量,"OOF残差共形(噪声, 常数宽度)",0.900,3838.15,1.000,3952.66,1.296
5,弹性模量,方差合成(分歧 噪声),0.901,3195.22,1.000,3095.00,1.014
6,弯曲强度,成员分位数(分歧),0.903,15.15,0.625,13.41,0.319
7,弯曲强度,"OOF残差共形(噪声, 常数宽度)",0.900,30.60,1.000,30.63,0.728
8,弯曲强度,方差合成(分歧 噪声),0.867,30.42,1.000,30.44,0.724
9,弯曲模量,成员分位数(分歧),0.968,640.11,0.625,717.54,0.233


In [9]:
# 单点预测示例测试
x = np.array([[310, 90, 30, 0.1, 1]])      # 喷嘴310℃, 热床90℃, 速度30, 层厚0.1mm, 方向1
print('特征顺序:', FEATS)
for tag in SHEETS.values():
    Pm = ENSEMBLES[tag].member_predictions(x)
    mu = ENSEMBLES[tag].predict(x)
    u = dispersion(Pm)
    lo, hi, _ = quadrature_conformal_interval(mu, u, CALIB[tag])
    q_res = conformal_quantile(np.abs(CAL[tag]['r']), ALPHA)
    sn = CALIB[tag]['sigma_noise']
    st = np.sqrt(u ** 2 + sn ** 2)

    print(f"{TAG_CN[tag]:6s} ({tag}): 点预测={mu[0]:7.2f} | "
          f"90%方差合成区间=[{lo[0]:7.2f}, {hi[0]:7.2f}] 宽={hi[0]-lo[0]:.2f} "
          f"(σ_model={u[0]:.2f}, σ_noise={sn:.2f}, σ_total={st[0]:.2f}, "
          f"常数残差半宽={q_res:.2f})")

特征顺序: ['T_noz', 'T_bed', 'speed', 'layer', 'direction']
拉伸强度   (TS): 点预测= 130.14 | 90%方差合成区间=[ 117.53,  142.74] 宽=25.21 (σ_model=3.86, σ_noise=3.07, σ_total=4.93, 常数残差半宽=11.05)
弹性模量   (TM): 点预测=14510.25 | 90%方差合成区间=[12956.78, 16063.72] 宽=3106.93 (σ_model=375.31, σ_noise=519.48, σ_total=640.87, 常数残差半宽=1976.33)
弯曲强度   (FS): 点预测= 165.65 | 90%方差合成区间=[ 153.31,  178.00] 宽=24.69 (σ_model=0.97, σ_noise=5.58, σ_total=5.67, 常数残差半宽=15.32)
弯曲模量   (FM): 点预测=9097.75 | 90%方差合成区间=[8441.48, 9754.01] 宽=1312.53 (σ_model=119.47, σ_noise=268.75, σ_total=294.11, 常数残差半宽=799.68)
冲击强度   (IS): 点预测=  26.81 | 90%方差合成区间=[  22.89,   30.74] 宽=7.85 (σ_model=0.97, σ_noise=1.81, σ_total=2.05, 常数残差半宽=4.14)


In [10]:
#结果导出
os.makedirs(OUT_DIR, exist_ok=True)
OUT_XLSX = os.path.join(OUT_DIR, INTERVAL_RESULT_XLSX)
with pd.ExcelWriter(OUT_XLSX) as writer:
    metric_table.to_excel(writer, sheet_name='指标汇总_方差合成', index=False)
    compare_table.to_excel(writer, sheet_name='三方法对比', index=False)
    pd.DataFrame([dict(目标=t, qhat=CALIB[t]['qhat'], sigma_noise=CALIB[t]['sigma_noise'],
                       alpha=CALIB[t]['alpha'], n_cal=CALIB[t]['n_cal'],
                       kappa=round(VD[t]['kappa'], 4),
                       噪声方差占比=round(float(VD[t]['noise_share']), 4))
                  for t in SHEETS.values()]).to_excel(
        writer, sheet_name='共形标定参数', index=False)
    for tag in SHEETS.values():
        e = EXT[tag]
        dct = {'cond': e['cond'].values, 'y_mean': e['y_mean'].values.astype(float),
               'pred': IV[tag][REC]['ex'][2], 'u': U_EXT[tag]}
        for m in METHODS:
            lo, hi, mu = IV[tag][m]['ex']
            dct[f'lo_{m}'] = lo; dct[f'hi_{m}'] = hi
        pd.DataFrame(dct).to_excel(writer, sheet_name=f'外部区间_{tag}', index=False)
    for tag in SHEETS.values():
        c = CAL[tag]
        lo, hi, mu = IV[tag][REC]['tr']
        pd.DataFrame(dict(cond=c['cond'], y=c['y'], oof_pred=mu, u=c['u'],
                          sigma_noise=c['sigma_noise'], r=c['r'], lo=lo, hi=hi)).to_excel(
            writer, sheet_name=f'训练OOF_{tag}', index=False)
print('已保存:', OUT_XLSX)


已保存: d:\Code_File\Python_Data_AI\Project_CFPA_Teacher\Data\基学习器_OOF残差复合预测区间\区间预测评估结果_复合.xlsx
